In [2]:
%pip install PyMuPDF


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\User\Desktop\WellDev Hackathon\RAG app\env\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\User\Desktop\WellDev Hackathon\RAG app\env\Scripts\python.exe -m pip install --upgrade pip' command.


In [5]:
%pip install ollama pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
%pip install sentence-transformers faiss-cpu PyMuPDF tiktoken


   ---------------------------------------- 0.0/884.8 kB ? eta -:--:--
   ----------------------- ---------------- 524.3/884.8 kB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 884.8/884.8 kB 3.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import ollama
import os
import fitz  # PyMuPDF
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List, Tuple, Dict
import logging
import sys
from dataclasses import dataclass
import tiktoken
import re
from tqdm import tqdm
from tqdm.autonotebook import tqdm, trange

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('rag_chain.log')
    ]
)

@dataclass
class Document:
    """Class to represent a document chunk with metadata"""
    content: str
    metadata: Dict
    embedding: np.ndarray = None

class DocumentChunker:
    """Class to handle document chunking with overlap"""
    
    def __init__(self, chunk_size: int = 512, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.tokenizer = tiktoken.get_encoding("cl100k_base")
        
    def _count_tokens(self, text: str) -> int:
        """Count the number of tokens in a text"""
        return len(self.tokenizer.encode(text))
    
    def create_chunks(self, text: str, metadata: Dict) -> List[Document]:
        """Split text into overlapping chunks"""
        chunks = []
        
        # Clean and normalize text
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Get token count for text
        tokens = self.tokenizer.encode(text)
        
        # Create chunks based on token count
        for i in range(0, len(tokens), self.chunk_size - self.chunk_overlap):
            # Get chunk tokens
            chunk_tokens = tokens[i:i + self.chunk_size]
            
            # Decode chunk tokens back to text
            chunk_text = self.tokenizer.decode(chunk_tokens)
            
            # Create document with metadata
            chunk_metadata = {
                **metadata,
                'chunk_index': len(chunks),
                'token_count': len(chunk_tokens)
            }
            
            chunks.append(Document(
                content=chunk_text,
                metadata=chunk_metadata
            ))
            
        return chunks

class VectorStore:
    """Class to handle document embeddings and similarity search"""
    
    def __init__(self, model_name: str = 'paraphrase-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.documents: List[Document] = []
        self.index = None
        
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to the vector store"""
        # Generate embeddings for all documents
        embeddings = self.model.encode([doc.content for doc in documents])
        
        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(embeddings)
        
        # Create FAISS index if it doesn't exist
        if self.index is None:
            self.index = faiss.IndexFlatIP(embeddings.shape[1])
            
        # Add embeddings to index
        self.index.add(embeddings.astype(np.float32))
        
        # Store documents with their embeddings
        for doc, embedding in zip(documents, embeddings):
            doc.embedding = embedding
            self.documents.append(doc)
    
    def similarity_search(self, query: str, k: int = 3) -> List[Tuple[Document, float]]:
        """Search for similar documents"""
        # Generate query embedding
        query_embedding = self.model.encode([query])
        faiss.normalize_L2(query_embedding)
        
        # Search index
        scores, indices = self.index.search(query_embedding.astype(np.float32), k)
        
        # Return documents and scores
        results = []
        for idx, score in zip(indices[0], scores[0]):
            results.append((self.documents[idx], float(score)))
            
        return results

class RAGChain:
    """Main RAG chain implementation with improved response generation"""
    
    def __init__(self, chunk_size: int = 512, chunk_overlap: int = 50):
        self.chunker = DocumentChunker(chunk_size, chunk_overlap)
        self.vector_store = VectorStore()
        
    def load_pdf_documents(self, pdf_folder: str) -> None:
        """Load and process PDF documents"""
        if not os.path.exists(pdf_folder):
            raise FileNotFoundError(f"PDF folder not found: {pdf_folder}")
            
        pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]
        if not pdf_files:
            raise ValueError(f"No PDF files found in {pdf_folder}")
        
        all_chunks = []
        for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
            pdf_path = os.path.join(pdf_folder, pdf_file)
            try:
                # Extract text from PDF
                with fitz.open(pdf_path) as doc:
                    text = " ".join(page.get_text() for page in doc)
                
                # Create metadata
                metadata = {
                    'source': pdf_file,
                    'path': pdf_path,
                    'type': 'pdf'
                }
                
                # Create chunks
                chunks = self.chunker.create_chunks(text, metadata)
                all_chunks.extend(chunks)
                
            except Exception as e:
                logging.error(f"Error processing {pdf_file}: {str(e)}")
                
        # Add all chunks to vector store
        self.vector_store.add_documents(all_chunks)
        logging.info(f"Processed {len(all_chunks)} chunks from {len(pdf_files)} PDFs")

    def _format_context(self, results: List[Tuple[Document, float]]) -> Tuple[str, str]:
        """Format retrieved context with improved structure"""
        prompt_context = []
        log_context = []
        
        for doc, score in results:
            # Format for prompt - keep it focused and relevant
            if score >= .7:  # Only include highly relevant context
                prompt_context.append(
                    f"Relevant Information (Confidence: {score:.2f}):\n"
                    f"{doc.content.strip()}\n"
                )
            
            # Format for logging/CSV
            log_context.append(
                f"Source: {doc.metadata['source']}\n"
                f"Relevance Score: {score:.4f}\n"
                f"Context Preview: {doc.content[:300]}...\n"
                f"{'='*50}"
            )
            
        return "\n".join(prompt_context), "\n".join(log_context)

    def generate_response(self, query: str) -> Tuple[str, str]:
        """Generate response using RAG with improved tone and length control"""
        # Retrieve relevant documents
        results = self.vector_store.similarity_search(query)
        
        # Format context
        prompt_context, log_context = self._format_context(results)
        
        # Generate response using Ollama
        prompt = f"""
        You are a friendly and professional customer service representative. Generate a concise yet helpful email response.

        Customer Query: "{query}"

        Available Information:
        {prompt_context}

        Guidelines:
        1. Keep the response between 100-150 words
        2. Use a warm, welcoming tone yet professional
        3. Show empathy and understanding
        4. Be clear and direct while remaining polite
        5. Only use information from the provided context
        6. If you can't fully answer based on the context, acknowledge this politely
        7. Include a clear next step or call to action

        Required Email Format:
        Subject: Re: [Brief, relevant subject]
        Dear Valued Customer,

        [2-3 sentences acknowledging the query and showing understanding]
        [1-2 sentences providing the answer or solution]
        [1 sentence for next steps or additional assistance]

        Best regards,
        Customer Support Team

        Example Response Format:
        Subject: Re: Order Tracking Update
        Dear Valued Customer,

        Thank you for reaching out about your order status. I understand your concern about the lack of tracking updates and I'm happy to help you with this.

        After checking your order details, I can confirm that your package is currently in transit and should arrive within 3-5 business days. We apologize for any worry this may have caused.

        Please don't hesitate to contact us if you need any further assistance or have additional questions.

        Best regards,
        Customer Support Team
        """

        try:
            response = ollama.chat(
                model="llama3.2",
                messages=[{"role": "user", "content": prompt}],
                options={
                    "temperature": 0.7,
                    "top_p": 0.9,
                    "max_tokens": 200
                }
            )
            
            # Validate and clean response
            email_response = response['message']['content']
            
            # Ensure response isn't too long
            paragraphs = email_response.split('\n\n')
            if len(paragraphs) > 6:  # If too many paragraphs
                # Keep essential parts only
                email_response = '\n\n'.join([
                    paragraphs[0],  # Subject
                    paragraphs[1],  # Greeting
                    paragraphs[2],  # Main content (acknowledgment)
                    paragraphs[3],  # Solution/answer
                    "Please don't hesitate to contact us if you need any further assistance.",
                    "Best regards,\nCustomer Support Team"
                ])
            
            return email_response, log_context
            
        except Exception as e:
            logging.error(f"Error generating response: {str(e)}")
            return f"Error generating response: {str(e)}", log_context

    def process_questions(self, input_csv: str, output_csv: str) -> None:
        """Process questions from CSV file"""
        try:
            df = pd.read_csv(input_csv)
            logging.info(f"Processing {len(df)} questions...")
            
            responses = []
            retrieval_contexts = []
            
            for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing questions"):
                response, context = self.generate_response(row['input'])
                responses.append(response)
                retrieval_contexts.append(context)

                # Print the generated response and context for each question
                print(f"Question: {row['input']}")
                print(f"Response: {response}")
                print(f"Context: {context}\n")
            
            df['actual_output'] = responses
            df['retrieval_context'] = retrieval_contexts
            df.to_csv(output_csv, index=False)
            logging.info(f"Responses saved to {output_csv}")
            
        except Exception as e:
            logging.error(f"Error processing questions: {str(e)}")
            raise

def main():
    try:
        # Initialize RAG chain with custom parameters
        rag_chain = RAGChain(
            chunk_size=512,  # Adjust based on your needs
            chunk_overlap=50  # Adjust based on your needs
        )
        
        # Load documents
        rag_chain.load_pdf_documents("Datasets/")
        
        # Process questions
        rag_chain.process_questions("Questions.csv", "Questions_with_responses.csv")
        
    except Exception as e:
        logging.error(f"Application error: {str(e)}")
        sys.exit(1)

if __name__ == "__main__":
    main()

Processing questions:  20%|██        | 1/5 [00:36<02:25, 36.47s/it]

Question: I received a damaged electronic item and noticed it after 3 days. However, your return policy mentions reporting damage within 48 hours. I'm a Platinum member in your loyalty program - what are my options?
Response: Subject: Re: Damaged Electronic Item - Platinum Member Assistance

Dear Valued Customer,

Thank you for bringing the damage to our attention, and I apologize for any inconvenience this has caused. Although your return policy mentions reporting damage within 48 hours, as a Platinum member, we value your loyalty and want to ensure that everything is resolved to your satisfaction.

Given the circumstances, I can offer you a replacement or store credit, whichever you prefer. Please reply to this email with your preferred option, and our team will guide you through the process.

Best regards,
Customer Support Team
Context: Source: ABC Company Return & Refund Policy.pdf
Relevance Score: 0.4846
Context Preview: Introduction At ABC Company, we understand that sometimes a 

Processing questions:  40%|████      | 2/5 [01:16<01:55, 38.37s/it]

Question: If I use multiple gift cards for a purchase and then need to return the item, how will the refund be processed? Also, can I split the refund between store credit and my original payment method?
Response: Subject: Re: Refund Process with Multiple Gift Cards

Dear Valued Customer,

Thank you for reaching out about your concerns regarding refunds using multiple gift cards. I understand that this can be a bit confusing, and I'm here to help clarify the process for you.

If you use multiple gift cards for a purchase and need to return the item, we will process the refund by applying the face value of each gift card in the order they were applied to the original purchase. Unfortunately, we cannot split the refund between store credit and your original payment method in this scenario.

If you have any further questions or concerns about your refund, please don't hesitate to contact us so we can assist you.

Best regards,
Customer Support Team
Context: Source: ABC Company Gift Card P

Processing questions:  60%|██████    | 3/5 [01:54<01:16, 38.14s/it]

Question: I run a small business and want to order bulk gift cards for my employees, but I also want to ensure our purchase aligns with our environmental commitments. What corporate gift card options do you offer, and how do they fit with your environmental policies?
Response: Subject: Re: Sustainable Corporate Gift Card Options

Dear Valued Customer,

Thank you for considering our corporate gift card options as part of your environmental commitment. We appreciate your dedication to sustainability and are happy to help.

Our corporate gift cards are made from fully recyclable materials and have minimal packaging. Additionally, we offer a carbon offset program that allows customers to calculate the carbon emissions associated with their purchases.

To learn more about our sustainable gift card options and how they align with our environmental policies, I'd be pleased to schedule a call with you at your convenience. Please let me know if this would be helpful in supporting your environme

Processing questions:  80%|████████  | 4/5 [02:34<00:39, 39.20s/it]

Question: I'm concerned about data privacy when joining your loyalty program. What customer data is collected, how is it used for personalization, and can I still earn points if I opt out of marketing communications?
Response: Subject: Re: Data Privacy Concerns with Loyalty Program

Dear Valued Customer,

Thank you for bringing your concerns about data privacy to our attention. We understand the importance of protecting your personal information and want to assure you that we take data protection seriously.

When joining our loyalty program, we collect basic customer data such as name, email address, and purchase history to ensure a personalized experience. This data is used to offer targeted promotions, rewards, and content. If you opt out of marketing communications, you will still be able to earn points on eligible purchases.

To review the full terms and conditions of our loyalty program, including our data collection and usage policies, please visit our website at [insert link]. W

Processing questions: 100%|██████████| 5/5 [03:11<00:00, 38.33s/it]

Question: As a seller wanting to join your marketplace, how do your environmental requirements affect my product listings and packaging? Are there specific eco-friendly standards I need to meet?
Response: Subject: Re: Joining Our Marketplace with Eco-Friendly Requirements

Dear Valued Seller,

Thank you for considering our marketplace as a platform for your business. I understand that meeting environmental requirements can be a concern, and I'm happy to provide guidance on our eco-friendly standards.

Our marketplace has adopted a set of eco-friendly guidelines for product listings and packaging, which include reducing single-use plastics, using biodegradable materials, and promoting sustainable practices. You can find more information on these standards in our Seller Guidelines document, which is available on our website.

If you have any questions or need further clarification, please don't hesitate to contact us.

Best regards,
Customer Support Team
Context: Source: ABC Company Mark